# LangChain Memory 시리즈 5/7: ConversationKnowledgeGraphMemory

## 학습 목표

이 노트북을 마치면 다음을 할 수 있어요.

- 지식 그래프(Knowledge Graph)가 엔티티 메모리와 어떻게 다른지 설명할 수 있어요
- 대화에서 주체-관계-객체 트리플(Triple)이 자동으로 추출되는 과정을 이해할 수 있어요
- `load_memory_variables({"input": 질문})`으로 관계 기반 검색을 수행할 수 있어요
- 조직도·팀 구조처럼 복잡한 관계망을 관리하는 시나리오에 적용할 수 있어요

## 사전 지식

| 개념 | 설명 |
|------|------|
| 지식 그래프 (Knowledge Graph) | 노드(엔티티)와 엣지(관계)로 구성된 그래프 형태의 지식 표현 |
| 트리플 (Triple) | (주체, 관계, 객체) 형태의 관계 표현 단위. 예: (민수, 일한다, 테크스타트) |
| ConversationEntityMemory | 엔티티 정보만 저장하는 메모리 (4번 노트북) |

## 엔티티 메모리 vs 지식 그래프 메모리

`ConversationEntityMemory`는 "민수"에 대한 정보를 텍스트로 저장해요. 반면 `ConversationKGMemory`는 민수-테크스타트-지영 같은 **관계 연결망** 자체를 그래프로 저장해요.

"지영과 함께 일하는 사람은?"처럼 관계 기반 질문에 강력한 능력을 발휘해요.

> 🔑 **핵심 개념**: EntityMemory는 "민수 = {직업: 백엔드, 회사: 테크스타트}"처럼 속성으로 저장하고, KGMemory는 "(민수) —[소속]→ (테크스타트)"처럼 관계 그래프로 저장합니다. 조직도·소셜 그래프 같은 관계 중심 데이터에 KG가 유리합니다.

> ⚠️ **자주 하는 실수**: `ConversationKGMemory`는 `inputs={"input": "..."}, outputs={"output": "..."}` 키를 사용해야 합니다. `"human"/"ai"` 키를 사용하면 엔티티 추출이 작동하지 않습니다.

```mermaid
flowchart LR
    subgraph EntityMem["EntityMemory (4번 노트북)"]
        E1["민수: 백엔드 개발자,<br/>테크스타트 소속"]
        E2["지영: 프론트엔드 개발자,<br/>테크스타트 소속"]
    end

    subgraph KGMem["KGMemory (이번 노트북)"]
        N1[민수]
        N2[지영]
        N3[테크스타트]
        N1 -- "같은팀" --> N2
        N1 -- "소속" --> N3
        N2 -- "소속" --> N3
    end

    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef storage fill:#e2d5f1,stroke:#6f42c1,color:#3d1f6e
    class N1,N2,N3 storage
    class E1,E2 process
```

> **참고**: 이 노트북은 레거시 메모리 클래스를 사용해요. LangChain 1.0.x에서는 그래프 DB 연동 + 관계 추출 체인 패턴이 권장돼요.

## 지식 그래프(Knowledge Graph)란 무엇인가요?

### 드라마 인물 관계도를 떠올려 보세요!

K-드라마를 볼 때 "인물 관계도"를 본 적 있나요? 등장인물들이 동그라미로 그려져 있고, 사이사이에 "친구", "연인", "라이벌" 같은 화살표가 연결되어 있죠. **지식 그래프(Knowledge Graph)가 바로 이것과 같은 구조예요!**

```
        [인물 관계도 = 지식 그래프]

    (민수) ──좋아한다──→ (지영)
      │                    │
    다닌다               다닌다
      │                    │
      ▼                    ▼
   (삼성)              (테크스타트)
```

### 트리플(Triple)이란?

지식 그래프는 **트리플(Triple)**이라는 아주 간단한 단위로 이루어져 있어요. 트리플은 **세 단어로 하나의 사실을 표현**하는 방법이에요:

> **(주체, 관계, 객체)** = "누가/무엇이" + "어떻게" + "누구를/무엇을"

| 트리플 | 한국어 해석 |
|:---|:---|
| (민수, 좋아한다, 지영) | "민수는 지영을 좋아한다" |
| (민수, 다닌다, 삼성) | "민수는 삼성에 다닌다" |
| (지영, 동료이다, 민수) | "지영은 민수의 동료이다" |
| (테크스타트, 위치하다, 강남) | "테크스타트는 강남에 위치한다" |

이런 트리플들이 모이면 **관계의 그물망(네트워크)**이 만들어져요. 이것이 바로 지식 그래프예요!

### EntityMemory vs KGMemory: 프로필 카드 vs 관계도

4번 노트북에서 배운 EntityMemory와 이번에 배울 KGMemory의 차이를 비유로 이해해 볼게요.

**EntityMemory = 프로필 카드**
- 각 사람에 대한 정보를 카드 한 장에 정리
- "민수: 백엔드 개발자, 파이썬 전문, 테크스타트 소속"
- 개인 정보는 잘 알 수 있지만, **사람 사이의 관계**는 파악하기 어려움

**KGMemory = 관계도 (인물 관계도)**
- 사람과 사람, 사람과 조직 사이의 **연결 관계**를 그래프로 저장
- "(민수) --[동료]--> (지영)", "(민수) --[소속]--> (테크스타트)"
- "지영과 함께 일하는 사람은?" 같은 **관계 기반 질문**에 강력함

```
[ EntityMemory - 프로필 카드 ]         [ KGMemory - 관계도 ]
┌──────────────────────┐         (민수)──[동료]──→(지영)
│ 민수                  │           │                │
│ 직업: 백엔드 개발자     │        [소속]          [소속]
│ 기술: 파이썬, Django   │           │                │
│ 회사: 테크스타트        │           ▼                ▼
└──────────────────────┘       (테크스타트)     (테크스타트)

→ "민수는 누구야?" ✅ 잘 답변    → "민수는 누구야?" ✅ 잘 답변
→ "민수의 동료는?" ❌ 어려움     → "민수의 동료는?" ✅ 지영!
```

이제 코드로 직접 지식 그래프를 만들어 볼게요!

In [ ]:
# ---------------------------------------------------
# 환경 변수 로드 및 지식 그래프 메모리 모듈 임포트
# ---------------------------------------------------
from dotenv import load_dotenv
# ⚠️ LangChain 1.0.x에서는 더 이상 `langchain.memory` 패키지를 사용하지 않습니다.
#    구식 메모리 클래스를 시연하기 위해 `langchain.memory`에서 불러옵니다.
from langchain.memory import ConversationKGMemory
# ChatOpenAI: 엔티티/관계 추출 LLM으로 사용
from langchain_openai import ChatOpenAI

load_dotenv()

## 1. 기본 사용법 - 지식 그래프 메모리 생성

`ConversationKGMemory`는 LLM을 사용하여 대화에서 엔티티와 관계를 추출합니다.


### 💡 지식 그래프 메모리 초기화

```mermaid
graph TD
    ENV["load_dotenv()"] --> LLM_INIT["ChatOpenAI(model='gpt-4o-mini', T=0)"]
    LLM_INIT --> MEM_INIT["ConversationKGMemory(llm=llm, return_messages=True)"]
    MEM_INIT --> READY[엔티티+관계 추출 준비]
```



In [ ]:
# ============================================================
# TODO: ConversationKGMemory를 생성하세요
# 힌트: ChatOpenAI(temperature=0) → ConversationKGMemory(llm=llm, return_messages=True)
# 예상 결과: save_context() 호출 시 (주체, 관계, 객체) 트리플이 그래프에 저장됨
# ============================================================

# 1단계: LLM 생성 (지식 그래프 구축에 사용)
# - temperature=0: 관계 추출 결과의 일관성 확보
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# 2단계: ConversationKGMemory 생성
# - return_messages=True: load_memory_variables()가 SystemMessage 형태로 반환
# - 입력/출력 키는 반드시 "input"/"output"이어야 엔티티 추출이 작동함


### 💡 관계 추출 파이프라인

```mermaid
flowchart TD
    INPUT1[대화 입력] --> SAVE1["save_context()"]
    SAVE1 --> GRAPH_HIST[internal history]
    GRAPH_HIST --> LLM1[LLM 호출로 엔티티/관계 추출]
    LLM1 --> KG_STORE[지식 그래프 노드/엣지 업데이트]
```



### 대화 저장 및 관계 추출

대화를 저장하면 엔티티 간의 관계가 자동으로 추출되어 지식 그래프에 저장됩니다.


### 💡 지식 그래프 질의 흐름

```mermaid
flowchart LR
    QUESTION["질문 입력 (예: 김민수씨는 누구?)"] --> LOAD1["load_memory_variables({'input': 질문})"]
    LOAD1 --> KG_QUERY[그래프 탐색 및 요약]
    KG_STORE[내부 KG 데이터] -.-> KG_QUERY
    KG_QUERY --> ANSWER[history 키에 그래프 기반 답변 반환]
```

In [ ]:
# ============================================================
# TODO: 관계 정보가 담긴 3개 대화를 save_context()로 저장하세요
# 힌트: inputs={"input": "..."}, outputs={"output": "..."} — "human"/"ai" 키는 불가
# 예상 결과: load_memory_variables({"input": "김민수씨는 누구입니까?"})로 그래프 탐색 가능
# ============================================================

# 1단계: 거주지 관계 저장
# - ⚠️ ConversationKGMemory는 "input"/"output" 키 사용 필수 ("human"/"ai" 키 사용 시 추출 불가)


# 2단계: 직업 관계 저장


# 3단계: 기술 스택 관계 저장


### 💡 조직 구조 그래프 구성 흐름

```mermaid
flowchart TD
    subgraph OrgChat
        STEP1[역할 설명 입력] --> SAVE_O1
        STEP2[팀원 관계 입력] --> SAVE_O2
        STEP3[협업 정보 입력] --> SAVE_O3
    end

    SAVE_O1 & SAVE_O2 & SAVE_O3 --> LLM_O[엔티티+관계 추출]
    LLM_O --> ORG_KG[org_memory 내부 그래프]
```



### 💡 조직 그래프 질의 루프

```mermaid
flowchart LR
    QUESTIONS[[질문 리스트]] --> ITER[for question in questions]
    ITER --> LOAD_ORG["org_memory.load_memory_variables({'input': question})"]
    LOAD_ORG --> ANSWER_ORG[history 기반 답변]
    ANSWER_ORG --> PRINT_ORG[질문/답변 출력]
```



### 저장된 지식 그래프 확인

`load_memory_variables()`를 사용하여 저장된 지식 그래프 정보를 확인할 수 있습니다.


### 💡 Entity vs KG 메모리 비교 흐름

```mermaid
flowchart TD
    INPUTCOMP[관계가 포함된 문장] --> SAVE_ENT[ConversationEntityMemory.save_context]
    INPUTCOMP --> SAVE_KG[ConversationKGMemory.save_context]

    SAVE_ENT --> ENT_STORE[entity_store.store]
    SAVE_KG --> KG_GRAPH[지식 그래프]

    ENT_STORE --> REPORT_ENT[엔티티 목록 출력]
    KG_GRAPH --> QUERY_KG["load_memory_variables({'input': 질문})"]
    QUERY_KG --> REPORT_KG[관계 포함 history 반환]
```



### EntityMemory vs KGMemory: 실제 결과 비교

위의 검색 결과를 보면 KGMemory의 특징이 명확해요. 같은 정보를 EntityMemory와 KGMemory가 어떻게 다르게 저장하는지 비교해 볼게요.

#### EntityMemory의 저장 방식 (프로필 카드)

```
entity_store = {
    "김민수씨": "김민수씨는 신입 개발자이고, 백엔드 개발자로
                 파이썬과 Django를 전문으로 한다."
}
```
- 김민수씨에 대한 정보가 **하나의 텍스트 덩어리**로 저장됨
- "김민수씨의 동료는?" → 텍스트에 동료 정보가 없으면 답변 불가

#### KGMemory의 저장 방식 (관계도)

```
(김민수씨) ──[is a]──→ (신입 개발자)
(김민수씨) ──[is part of]──→ (우리 회사)
(김민수씨) ──[전문으로 하는]──→ (파이썬)
(김민수씨) ──[전문으로 하는]──→ (Django)
(김민수씨) ──[is a]──→ (백엔드 개발자)
```
- 각 정보가 **관계(화살표)로 연결**되어 있음
- "김민수씨가 전문으로 하는 것은?" → 그래프에서 "전문으로 하는" 관계를 따라가면 바로 찾음!

#### 언제 어떤 메모리를 사용할까?

| 사용 상황 | 추천 메모리 | 이유 |
|:---|:---:|:---|
| 고객 상담 / CRM | EntityMemory | 고객 프로필 정보 관리에 적합 |
| 조직도 / 팀 구조 | **KGMemory** | 상사-부하, 팀-멤버 관계 추적에 강력 |
| 소셜 네트워크 분석 | **KGMemory** | 사람 간 관계(친구, 팔로워) 탐색에 적합 |
| 제품 카탈로그 | EntityMemory | 제품별 속성(가격, 색상) 관리에 적합 |
| 프로젝트 의존성 관리 | **KGMemory** | 프로젝트-담당자-기술 연결 관계 파악에 유리 |
| 간단한 FAQ 챗봇 | EntityMemory | 복잡한 관계가 필요 없어 EntityMemory가 효율적 |

> **핵심 판단 기준**: "A는 B와 어떤 관계야?"처럼 **관계를 묻는 질문**이 많다면 KGMemory, "A에 대해 알려줘"처럼 **속성을 묻는 질문**이 많다면 EntityMemory가 적합해요.

In [ ]:
# ---------------------------------------------------
# 지식 그래프 기반 관계 검색
# ---------------------------------------------------
# load_memory_variables({"input": 질문}): 질문과 관련된 엔티티를 그래프에서 탐색
# 결과는 SystemMessage.content에 "(엔티티) is/has/전치사 (객체)" 형태의 트리플 텍스트로 반환
memory_vars = memory.load_memory_variables({"input": "김민수씨는 누구입니까?"})

print("=" * 60)
print("🔍 지식 그래프 기반 메모리 검색")
print("=" * 60)
print(f"\n질문: 김민수씨는 누구입니까?")
print(f"\n관련 정보:")
print(memory_vars.get("history", "정보 없음"))

## 2. 실전 예제: 조직 구조 관리

팀원 간의 관계와 역할을 추적하는 조직 구조 관리 예제입니다.


In [ ]:
# ============================================================
# TODO: 조직 구조 메모리를 생성하고 5개의 조직 관련 대화를 저장하세요
# 힌트: ConversationKGMemory(llm=llm, return_messages=True) → for 루프로 저장
# 예상 결과: 박지영, 이수진, 최민호 간 역할/협업 관계가 그래프에 저장됨
# ============================================================

# 1단계: 조직 구조 메모리 생성


# 2단계: 조직 구조 대화 데이터 (역할과 협업 관계 포함)
org_conversations = [
    {
        "input": "박지영은 우리 회사의 디자인 팀장입니다.",
        "output": "알겠습니다. 박지영님의 역할을 기록했습니다."
    },
    {
        "input": "이수진은 박지영 팀장의 팀원이고, UI/UX 디자이너입니다.",
        "output": "이수진님이 디자인 팀의 UI/UX 디자이너로 기록되었습니다."
    },
    {
        "input": "최민호는 개발 팀의 리드 개발자입니다.",
        "output": "최민호님이 개발 팀의 리드 개발자로 기록되었습니다."
    },
    {
        "input": "박지영 팀장과 최민호 리드 개발자는 프로젝트 회의에서 자주 협업합니다.",
        "output": "협업 관계를 기록했습니다."
    },
    {
        "input": "이수진은 최민호와 함께 모바일 앱 프로젝트를 진행하고 있습니다.",
        "output": "프로젝트 협업 관계를 기록했습니다."
    }
]

# 3단계: 모든 대화 저장 (각 저장마다 LLM이 관계 트리플을 추출하여 그래프 업데이트)


In [ ]:
# ---------------------------------------------------
# 조직 구조 지식 그래프 질의 응답
# ---------------------------------------------------
# 질문에 언급된 엔티티와 연결된 그래프 노드를 탐색하여 관련 관계 정보를 반환
questions = [
    "박지영은 누구입니까?",
    "이수진은 어떤 역할을 하나요?",
    "최민호와 협업하는 사람은 누구인가요?",
    "모바일 앱 프로젝트에 참여하는 사람은 누구인가요?"
]

print("=" * 60)
print("👥 조직 구조 지식 그래프 검색")
print("=" * 60)

for question in questions:
    memory_vars = org_memory.load_memory_variables({"input": question})
    print(f"\n❓ 질문: {question}")
    print(f"📋 답변: {memory_vars.get('history', '정보 없음')}")
    print("-" * 60)

### 지식 그래프 시각화

Pyvis를 사용하여 내부 지식 그래프를 인터랙티브하게 시각화합니다.
노드를 드래그하고, 확대/축소하여 엔티티 간 관계를 탐색할 수 있습니다.

In [ ]:
# ---------------------------------------------------
# 조직 구조 지식 그래프 인터랙티브 시각화 (Pyvis)
# ---------------------------------------------------
import html as _html
from pyvis.network import Network
from IPython.display import HTML, display

G = org_memory.kg._graph

net = Network(
    height="500px",
    width="100%",
    directed=True,
    notebook=False,
    cdn_resources="remote",
)

net.from_nx(G)

for node in net.nodes:
    node["font"] = {"size": 16}
    node["size"] = 25

for edge in net.edges:
    rel = edge.get("label") or edge.get("title") or ""
    edge["title"] = rel
    edge["label"] = rel
    edge["font"] = {"size": 12, "align": "middle"}

net.set_options("""
{
  "physics": {
    "forceAtlas2Based": {
      "gravitationalConstant": -100,
      "springLength": 200
    },
    "solver": "forceAtlas2Based"
  }
}
""")

net.save_graph("org_knowledge_graph.html")
with open("org_knowledge_graph.html", "r") as f:
    graph_html = f.read()

display(HTML(
    f'<iframe srcdoc="{_html.escape(graph_html)}" '
    f'width="100%" height="520" frameborder="0"></iframe>'
))

In [ ]:
# ---------------------------------------------------
# 김민수씨 지식 그래프 시각화
# ---------------------------------------------------
G_basic = memory.kg._graph

net2 = Network(
    height="400px",
    width="100%",
    directed=True,
    notebook=False,
    cdn_resources="remote",
)

net2.from_nx(G_basic)

for node in net2.nodes:
    node["font"] = {"size": 16}
    node["size"] = 25

for edge in net2.edges:
    rel = edge.get("label") or edge.get("title") or ""
    edge["title"] = rel
    edge["label"] = rel
    edge["font"] = {"size": 12, "align": "middle"}

net2.set_options("""
{
  "physics": {
    "forceAtlas2Based": {
      "gravitationalConstant": -100,
      "springLength": 200
    },
    "solver": "forceAtlas2Based"
  }
}
""")

net2.save_graph("basic_knowledge_graph.html")
with open("basic_knowledge_graph.html", "r") as f:
    graph_html2 = f.read()

display(HTML(
    f'<iframe srcdoc="{_html.escape(graph_html2)}" '
    f'width="100%" height="420" frameborder="0"></iframe>'
))

## 3. ConversationEntityMemory와의 차이

`ConversationEntityMemory`는 엔티티 정보만 저장하지만,  
`ConversationKGMemory`는 엔티티 간의 **관계**도 함께 저장합니다.


In [ ]:
# ---------------------------------------------------
# ConversationEntityMemory vs ConversationKGMemory 비교
# ---------------------------------------------------
# 동일 문장에서 EntityMemory는 엔티티 속성 텍스트를, KGMemory는 관계 트리플을 저장
from langchain.memory import ConversationEntityMemory

# 비교용 문장 (관계 정보 포함)
comparison_input = "김철수는 박영희의 상사이고, 박영희는 이민호의 멘토입니다."

# 1단계: EntityMemory에 저장 → 엔티티별 요약 텍스트로 보관


# 2단계: KGMemory에 저장 → (주체, 관계, 객체) 트리플로 그래프에 보관


# 비교


## 4. 실전 예제: ConversationChain + KG 메모리 채팅

`ConversationChain`에 `ConversationKGMemory`를 연결하면, 대화할 때마다 관계 그래프가 자동으로 구축됩니다.
질문 시 그래프에서 관련 관계를 탐색하여 프롬프트의 `{history}` 변수에 주입합니다.

```mermaid
flowchart LR
    USER[사용자 입력] --> CHAIN["conversation.predict()"]
    CHAIN --> LOAD["① load_memory_variables()<br/>그래프 탐색 → 관련 트리플 반환"]
    LOAD --> LLM["② LLM 호출<br/>관계 정보가 프롬프트에 주입"]
    LLM --> SAVE["③ save_context()<br/>새 관계 트리플 추출 → 그래프 업데이트"]
    SAVE --> RESP[AI 응답]
```

In [ ]:
# ---------------------------------------------------
# ConversationChain + KGMemory 채팅 구성
# ---------------------------------------------------
from langchain.chains import ConversationChain
from langchain.prompts.prompt import PromptTemplate

# KG 메모리 전용 프롬프트
# - {history}: 그래프에서 탐색한 관련 관계 트리플이 주입됨
# - {input}: 현재 사용자 입력
template = """다음은 사람과 AI 어시스턴트의 대화입니다.
AI는 관련 정보(Relevant Information)에 포함된 내용만 활용하여 응답합니다.
정보가 없으면 모른다고 솔직하게 답합니다.

Relevant Information:

{history}

Conversation:
Human: {input}
AI:"""

prompt = PromptTemplate(input_variables=["history", "input"], template=template)

# ConversationChain 생성 (KGMemory 연동)
chat_memory = ConversationKGMemory(llm=llm)
conversation_kg = ConversationChain(
    llm=llm,
    prompt=prompt,
    memory=chat_memory,
)

In [ ]:
# ============================================================
# TODO: conversation_kg.predict()로 팀 구성원과 역할 관계를 입력하세요
# 힌트: conversation_kg.predict(input="우리 팀을 소개할게요. 팀장 한소희는...")
# 예상 결과: AI가 팀 구성에 대해 응답하고, KG에 관계 트리플이 저장됨
# ============================================================

# 여기에 코드를 작성하세요
pass

In [ ]:
# ============================================================
# TODO: conversation_kg.predict()로 프로젝트와 협업 관계를 추가하세요
# 힌트: conversation_kg.predict(input="한소희와 정우성은 '추천 엔진 v2' 프로젝트를...")
# 예상 결과: 프로젝트-담당자 관계가 그래프에 추가됨
# ============================================================

# 여기에 코드를 작성하세요
pass

In [ ]:
# ============================================================
# TODO: conversation_kg.predict()로 관계 기반 질문을 하여 맥락 유지를 확인하세요
# 힌트: conversation_kg.predict(input="정우성이 담당하는 업무와 관련된 사람들을 정리해줘.")
# 예상 결과: AI가 정우성의 역할(MLOps), 상사(한소희), 프로젝트 정보를 기억하며 응답
# ============================================================

# 여기에 코드를 작성하세요
pass

In [ ]:
# ---------------------------------------------------
# 채팅을 통해 구축된 지식 그래프 확인
# ---------------------------------------------------
# chat_memory.kg._graph: 내부 NetworkX 그래프 객체
print("=" * 60)
print("🔗 채팅으로 구축된 지식 그래프 트리플")
print("=" * 60)

G_chat = chat_memory.kg._graph
for subj, obj, data in G_chat.edges(data=True):
    rel = data.get("relation", "관계없음")
    print(f"  ({subj}) —[{rel}]→ ({obj})")

In [ ]:
# ---------------------------------------------------
# 채팅으로 구축된 지식 그래프 시각화
# ---------------------------------------------------
import html as _html
from pyvis.network import Network
from IPython.display import HTML, display

G_chat = chat_memory.kg._graph

net_chat = Network(
    height="500px",
    width="100%",
    directed=True,
    notebook=False,
    cdn_resources="remote",
)

net_chat.from_nx(G_chat)

for node in net_chat.nodes:
    node["font"] = {"size": 16}
    node["size"] = 25

for edge in net_chat.edges:
    rel = edge.get("label") or edge.get("title") or ""
    edge["title"] = rel
    edge["label"] = rel
    edge["font"] = {"size": 12, "align": "middle"}

net_chat.set_options("""
{
  "physics": {
    "forceAtlas2Based": {
      "gravitationalConstant": -100,
      "springLength": 200
    },
    "solver": "forceAtlas2Based"
  }
}
""")

net_chat.save_graph("chat_knowledge_graph.html")
with open("chat_knowledge_graph.html", "r") as f:
    graph_html_chat = f.read()

display(HTML(
    f'<iframe srcdoc="{_html.escape(graph_html_chat)}" '
    f'width="100%" height="520" frameborder="0"></iframe>'
))

## 핵심 정리

```python
# ConversationKGMemory 기본 사용법
from langchain.memory import ConversationKGMemory
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
memory = ConversationKGMemory(llm=llm, return_messages=True)

# 대화 저장 (엔티티와 관계 자동 추출)
memory.save_context(
    inputs={"input": "사용자 메시지"},
    outputs={"output": "AI 응답"}
)

# 관계 정보 검색
memory_vars = memory.load_memory_variables({"input": "엔티티에 대한 질문"})
```

### 주요 특징
- ✅ **관계 기반 저장**: 엔티티 간의 관계를 그래프 형태로 저장
- ✅ **복잡한 연결망 이해**: 여러 엔티티 간의 복잡한 관계를 이해하고 활용
- ✅ **맥락 보존**: 관계 정보를 기반으로 대화 맥락을 유지
- ✅ **효율적 검색**: 관련 엔티티와 관계를 효율적으로 검색

### ConversationEntityMemory와의 차이

| 특징 | ConversationEntityMemory | ConversationKGMemory |
|------|-------------------------|---------------------|
| 저장 내용 | 엔티티 정보만 | 엔티티 + 관계 정보 |
| 구조 | 플랫한 구조 | 그래프 구조 |
| 관계 추적 | 제한적 | 강력함 |
| 복잡한 관계 | 처리 어려움 | 처리 가능 |
| 적합한 경우 | 단순한 엔티티 정보 | 복잡한 관계망 |

### 주의사항
- ⚠️ **LLM 필요**: 지식 그래프 구축을 위해 LLM 인스턴스가 필요함
- ⚠️ **비용**: 관계 추출에 LLM 호출이 필요하므로 비용 발생
- ⚠️ **복잡도**: 단순한 경우에는 ConversationEntityMemory가 더 효율적일 수 있음
- ⚠️ **대화형 사용**: ConversationChain과 함께 사용하는 것이 일반적 (deprecated 경고 있음)